# Work Done — Pushing a Block

A constant force $F$ pushes a block of mass $m$ along a frictionless floor (same set-up as the Newton's second law demo). The **work done** by a force is

$$W = Fs\cos\theta$$

and here the force points along the motion, so $\cos\theta = 1$ and it's simply

$$W = Fs$$

— the force multiplied by the distance moved **in the direction of the force**.

Press **Run** and watch the readout: as the block covers distance $s$, the work done $W = Fs$ grows in proportion. Since the floor is frictionless, every joule of work goes into the block's **kinetic energy**:

$$W = Fs = \tfrac{1}{2}mv^2 = \Delta KE$$

This is the **work–energy theorem**: the work done on the block equals its gain in kinetic energy. Check the readout — $W$ and $KE$ agree at every instant.

**Things to try:**

- Double $F$ → double the work done over the same distance (and the block gets there faster: $a = F/m$).
- Bigger $m$ → the same $W = Fs$ for the same distance, but a slower block — the same work buys the same KE, just delivered later.


In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle, FancyArrowPatch

from matplotlib.animation import FuncAnimation

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    %pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display, HTML


TRACK = 21.5   # length of the visible track the block travels (m)


def build_block_figure(x=0.0, F=100, m=20, a=5.0):
    """Draw the block, the force arrow and the acceleration arrow at position x."""
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.set_xlim(0, 25)
    ax.set_ylim(-1, 6)
    ax.axhline(1, color="black", linewidth=2)

    block = Rectangle((x, 1), 2.5, 1.6, facecolor="lightblue",
                      edgecolor="black", linewidth=2)
    ax.add_patch(block)
    mass_text = ax.text(x + 1.25, 1.8, f"{m} kg", ha="center", va="center",
                        fontsize=13)

    force_arrow = FancyArrowPatch((x + 2.7, 1.8), (x + 5.7, 1.8),
                                  arrowstyle="-|>", mutation_scale=20,
                                  linewidth=2, color="black")
    ax.add_patch(force_arrow)
    force_text = ax.text(x + 4.2, 2.25, f"F = {F} N", ha="center", fontsize=12)

    accel_arrow = FancyArrowPatch((x + 0.2, 4.1), (x + 4.2, 4.1),
                                  arrowstyle="-|>", mutation_scale=22,
                                  linewidth=2.5, color="firebrick")
    ax.add_patch(accel_arrow)
    accel_text = ax.text(x + 2.2, 4.65, f"a = {a:.2f} m/s²",
                         ha="center", fontsize=13, fontweight="bold")

    title = ax.set_title("", fontsize=12)

    ax.set_xlabel("Position (m)")
    ax.set_yticks([])
    for side in ("left", "right", "top"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()

    artists = dict(
        block=block, mass_text=mass_text,
        force_arrow=force_arrow, force_text=force_text,
        accel_arrow=accel_arrow, accel_text=accel_text,
        title=title,
    )
    return fig, artists


def readout(F, m, t):
    """Distance, speed, work done and KE after time t under force F."""
    a = F / m
    s = 0.5 * a * t**2
    v = a * t
    W = F * s
    KE = 0.5 * m * v**2
    return a, s, v, W, KE


def work_animation(F, m, n_frames=35):
    """In-browser animation: the block accelerates; W = Fs grows with distance."""
    a = F / m
    t_end = np.sqrt(2 * TRACK / a)         # ends as the block reaches the edge
    times = np.linspace(0, t_end, n_frames)

    fig, art = build_block_figure(x=0.0, F=F, m=m, a=a)

    def update(i):
        a_, s, v, W, KE = readout(F, m, times[i])
        art["block"].set_x(s)
        art["mass_text"].set_position((s + 1.25, 1.8))
        art["force_arrow"].set_positions((s + 2.7, 1.8), (s + 5.7, 1.8))
        art["force_text"].set_position((s + 4.2, 2.25))
        art["accel_arrow"].set_positions((s + 0.2, 4.1), (s + 4.2, 4.1))
        art["accel_text"].set_position((s + 2.2, 4.65))
        art["title"].set_text(
            f"s = {s:,.1f} m      v = {v:,.1f} m/s      "
            f"W = Fs = {W:,.0f} J   =   KE = ½mv² = {KE:,.0f} J")
        return tuple(art.values())

    anim = FuncAnimation(fig, update, frames=n_frames, interval=60,
                         repeat=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def show_start(F, m):
    """Static preview: block at the start line."""
    a = F / m
    fig, art = build_block_figure(x=0.0, F=F, m=m, a=a)
    art["title"].set_text("Press Run — then watch W = Fs grow with distance")
    display(fig)
    plt.close(fig)


def summary(F, m):
    a = F / m
    W_full = F * TRACK
    v_end = np.sqrt(2 * a * TRACK)
    return (f"<b>F = {F} N</b>, m = {m} kg &nbsp;→&nbsp; a = F/m = "
            f"<b>{a:.2f} m/s²</b><br>"
            f"Work done over the full {TRACK:g} m track: "
            f"W = Fs = ({F})({TRACK:g}) = <b>{W_full:,.0f} J</b><br>"
            f"…all of it becomes KE: v = √(2as) = {v_end:,.1f} m/s, so "
            f"KE = ½mv² = <b>{W_full:,.0f} J</b> ✓")


In [ ]:
# ============================================================
# Work done: vary F and m, then watch W = Fs build up
# ============================================================

force_slider = widgets.IntSlider(
    value=100, min=20, max=300, step=20,
    description="Force F (N):",
    continuous_update=False
)

mass_slider = widgets.IntSlider(
    value=20, min=5, max=60, step=5,
    description="Mass m (kg):",
    continuous_update=False
)

run_button = widgets.Button(description="Run", button_style="primary", icon="play")
reset_button = widgets.Button(description="Reset")
value_label = widgets.HTML()
sim_output = widgets.Output(layout=widgets.Layout(width="700px"))


def show_initial():
    value_label.value = summary(force_slider.value, mass_slider.value)
    show_start(force_slider.value, mass_slider.value)


def run_clicked(button):
    value_label.value = summary(force_slider.value, mass_slider.value)
    with sim_output:
        sim_output.clear_output(wait=True)
        display(work_animation(force_slider.value, mass_slider.value))


def reset_clicked(button):
    with sim_output:
        sim_output.clear_output(wait=True)
        show_initial()


def slider_changed(change):
    with sim_output:
        sim_output.clear_output(wait=True)
        show_initial()


run_button.on_click(run_clicked)
reset_button.on_click(reset_clicked)
force_slider.observe(slider_changed, names="value")
mass_slider.observe(slider_changed, names="value")

controls = widgets.VBox([
    widgets.HTML("<h3>Controls</h3>"),
    force_slider,
    mass_slider,
    widgets.HBox([run_button, reset_button]),
    value_label
], layout=widgets.Layout(width="340px"))

display(widgets.HBox([controls, sim_output],
                     layout=widgets.Layout(align_items="flex-start", gap="18px")))

with sim_output:
    show_initial()
